In [1]:
import numpy as np
import os
import pandas as pd
import cv2
import shutil

# Pairing each value and corresponding subtype in masks

In [2]:
# Paths
mask_dir = '/kaggle/input/rasouldatav2/HemoricaV1-masks'
image_dir = '/kaggle/input/rasouldatav2/HemoricaV1-CT'
output_label_dir = '/kaggle/working/Working/labels'
output_image_dir = '/kaggle/working/Working/images'
csv_path = '/kaggle/input/rasouldatav2/HemoricaV1-dataset.csv'

In [3]:
# Load CSV
df = pd.read_csv(csv_path)

# Track which unique non-zero values we've already processed
processed_values = set()

# Hemorrhage subtype columns
subtype_cols = ['ICH', 'IVH', 'EPH', 'SDH', 'SAH']

# Loop over .npy files
for filename in os.listdir(mask_dir):
    if not filename.endswith('.npy'):
        continue

    file_path = os.path.join(mask_dir, filename)

    try:
        array = np.load(file_path)
        unique_vals = np.unique(array)

        # Check if exactly two unique values and one is zero
        if len(unique_vals) == 2 and 0 in unique_vals:
            nonzero_val = unique_vals[unique_vals != 0][0]

            # Skip if we've already processed this value
            if nonzero_val in processed_values:
                continue

            processed_values.add(nonzero_val)

            # Parse filename to get patient_name and slice_number
            parts = filename.replace('.npy', '').split('_')
            if len(parts) < 2:
                print(f"⚠️ Unexpected filename format: {filename}")
                continue

            slice_number = int(parts[-1])
            patient_name = '_'.join(parts[:-1])

            # Fix spacing in patient_name if needed
            patient_name = patient_name.replace('_', ' ')

            # Find row in CSV
            match = df[
                (df['patient_name'] == patient_name) &
                (df['slice_number'] == slice_number)
            ]

            if match.empty:
                print(f"❌ No CSV match for: {patient_name}, slice {slice_number}")
                continue

            row = match.iloc[0]

            # Check which subtype columns are non-zero
            active_subtypes = [col for col in subtype_cols if row[col] != 0]

            if active_subtypes:
                print(f"\n✅ Unique value {nonzero_val} corresponds to subtypes: {active_subtypes}")
            else:
                print(f"\nℹ️ Unique value {nonzero_val} has no hemorrhage subtype (all 0)")

    except Exception as e:
        print(f"⚠️ Error reading {filename}: {e}")



✅ Unique value 1 corresponds to subtypes: ['ICH']

✅ Unique value 3 corresponds to subtypes: ['EPH']

✅ Unique value 4 corresponds to subtypes: ['SDH']

✅ Unique value 2 corresponds to subtypes: ['IVH']

✅ Unique value 5 corresponds to subtypes: ['SAH']


In [4]:
# === Class ID to Name Mapping ===
CLASS_ID_TO_NAME = {
    0: 'ICH',
    1: 'IVH',
    2: 'EPH',
    3: 'SDH',
    4: 'SAH',
}

def save_class_names(output_label_dir):
    classes_path = os.path.join(output_label_dir, "classes.txt")
    with open(classes_path, 'w') as f:
        for class_id in sorted(CLASS_ID_TO_NAME.keys()):
            f.write(f"{CLASS_ID_TO_NAME[class_id]}\n")
    print(f"Saved class names to: {classes_path}")

# Generating annotations

In [5]:
def mask_to_obb_annotations(mask, img_shape, min_size_ratio=0.005):
    """
    Extracts oriented bounding boxes (OBBs) from a segmentation mask.
    Returns annotations in YOLOv8 OBB format: class_id x1 y1 x2 y2 x3 y3 x4 y4
    where coordinates are normalized to [0, 1].
    """
    height, width = img_shape
    min_size = min_size_ratio * min(width, height)
    annotations = []

    for value in np.unique(mask):
        if value == 0:
            continue  # skip background

        class_id = value - 1
        binary_mask = (mask == value).astype(np.uint8)
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            rect = cv2.minAreaRect(cnt)
            (cx, cy), (w, h), angle = rect

            if w < min_size or h < min_size:
                continue

            box = cv2.boxPoints(rect)
            box = np.clip(box, 0, [width - 1, height - 1])  # ensure inside image bounds
            box_norm = [(x / width, y / height) for x, y in box]
            flat_coords = [coord for point in box_norm for coord in point]

            annotations.append((class_id, flat_coords))

    return annotations


In [6]:
# === YOLOv8 OBB Annotation Writer ===
def generate_yolo_annotations_per_class(image_id, mask, img_width, img_height, output_label_dir):
    obb_annotations = mask_to_obb_annotations(mask, (img_height, img_width))  # Note: (H, W)

    annotation_file = os.path.join(output_label_dir, f"{image_id}.txt")

    if not obb_annotations:
        with open(annotation_file, 'w') as f:
            f.write("")  # Empty file
        print(f"No objects detected for {image_id}.")
        return

    with open(annotation_file, 'w') as f:
        for class_id, coords in obb_annotations:
            coord_str = ' '.join(f"{c:.6f}" for c in coords)
            f.write(f"{int(class_id)} {coord_str}\n")

    class_names_in_image = {CLASS_ID_TO_NAME.get(int(class_id), 'Unknown') for class_id, _ in obb_annotations}
    print(f"✅ Saved {len(obb_annotations)} OBB annotations for {image_id}. Classes: {', '.join(class_names_in_image)}")


In [7]:
# === Dataset Processing ===
def process_dataset(mask_dir, image_dir, output_label_dir):
    os.makedirs(output_label_dir, exist_ok=True)
    save_class_names(output_label_dir)

    for mask_filename in os.listdir(mask_dir):
        if not mask_filename.endswith(".npy"):
            continue

        mask_path = os.path.join(mask_dir, mask_filename)
        mask = np.load(mask_path)

        image_filename = mask_filename
        image_path = os.path.join(image_dir, image_filename)

        if not os.path.exists(image_path):
            print(f"Skipping {mask_filename} - Image file not found.")
            continue

        img = np.load(image_path)
        if img is None or img.size == 0:
            print(f"Skipping {mask_filename} - Invalid image.")
            continue

        img_height, img_width = img.shape[:2]
        image_id = os.path.splitext(mask_filename)[0]

        generate_yolo_annotations_per_class(image_id, mask, img_width, img_height, output_label_dir)

    print("✅ YOLO annotation files created successfully.")

In [8]:
process_dataset(mask_dir, image_dir, output_label_dir)

Saved class names to: /kaggle/working/Working/labels/classes.txt
No objects detected for 61_13.
No objects detected for BAJELAN.MOJTABA-638573343932978246_2.
No objects detected for AHMADI MEHRNOSH-638573279783984529_29.
✅ Saved 1 OBB annotations for PORFARHADI MOHAMAD JAVAD-638573325545960527_20. Classes: ICH
✅ Saved 3 OBB annotations for TOFIGHI MOHAMMAD-638573336274598086_15. Classes: ICH, IVH
No objects detected for NORAEI SHAHINDOKHT-638839486874254842_15.
No objects detected for BOLANDHEMAT ALI.MOHAMAD-638573387891883483_4.
No objects detected for 67_29.
No objects detected for HAMZEEI FARIDE-638573365169189689_26.
✅ Saved 1 OBB annotations for SHAMSAEI MEHDI-638839513134388151_20. Classes: EPH
No objects detected for PARSEH DAVOD-638839478192330005_36.
No objects detected for 13_40.
No objects detected for POURMOMEN MARZIYEH-638573305944410911_35.
No objects detected for ASGARIYAN MOHAMADHOSSEIN-638573324701702776_19.
No objects detected for NIKFARMAN NARGES-638573270048500219_3

In [9]:
!zip -r /kaggle/working/Rasoul_allmulticlasslabels.zip '/kaggle/working/Working/labels'

  adding: kaggle/working/Working/labels/ (stored 0%)
  adding: kaggle/working/Working/labels/SHAHMIRZAI PARVIN-638839450246490138_19.txt (deflated 27%)
  adding: kaggle/working/Working/labels/RAHMATI MOHAMAD HOSEIN-638573324067767316_20.txt (deflated 47%)
  adding: kaggle/working/Working/labels/LASHGARI AZAM-638839461907107237_21.txt (deflated 27%)
  adding: kaggle/working/Working/labels/53_29.txt (stored 0%)
  adding: kaggle/working/Working/labels/SHARIATI MASOOD-638573299427214142_28.txt (stored 0%)
  adding: kaggle/working/Working/labels/57_18.txt (stored 0%)
  adding: kaggle/working/Working/labels/KOORHASANLOO GHAHREMAN-638573354401162598_1.txt (stored 0%)
  adding: kaggle/working/Working/labels/JAMSHIDI FATEMEH-638573375988317395_8.txt (deflated 46%)
  adding: kaggle/working/Working/labels/MADANI GHOLAMREZA-638573381463136795_9.txt (deflated 27%)
  adding: kaggle/working/Working/labels/21_13.txt (stored 0%)
  adding: kaggle/working/Working/labels/BABAYEI NAZANINZAHRA-6388395010663

# Converting to .png

In [10]:
def apply_windowing(image, window_center=40, window_width=80):
    """Apply windowing to CT image using center and width."""
    lower = window_center - (window_width // 2)
    upper = window_center + (window_width // 2)
    image = np.clip(image, lower, upper)
    image = ((image - lower) / (upper - lower)) * 255.0
    return image.astype(np.uint8)

In [11]:
def extract_skull(image):
    _, thresh = cv2.threshold(image, 1, 255, cv2.THRESH_BINARY)
    kernel = np.ones((3, 3), np.uint8)
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(cleaned, connectivity=8)
    if num_labels < 2:
        return cleaned
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest_label).astype(np.uint8) * 255

In [12]:
def convert_npy_to_png(npy_dir, output_dir, window_center=40, window_width=80):
    os.makedirs(output_dir, exist_ok=True)
    
    for filename in os.listdir(npy_dir):
        if filename.endswith(".npy"):
            image = np.load(os.path.join(npy_dir, filename))
            
            if image.ndim > 2:
                image = image[:, :, 0]
                
            windowed = apply_windowing(image, window_center, window_width)
            png_path = os.path.join(output_dir, filename.replace(".npy", ".png"))
            cv2.imwrite(png_path, windowed)
            print(f"✅ Converted: {filename}")

In [13]:
convert_npy_to_png(image_dir, output_image_dir, window_center=40, window_width=80)

✅ Converted: 61_13.npy
✅ Converted: BAJELAN.MOJTABA-638573343932978246_2.npy
✅ Converted: AHMADI MEHRNOSH-638573279783984529_29.npy
✅ Converted: PORFARHADI MOHAMAD JAVAD-638573325545960527_20.npy
✅ Converted: TOFIGHI MOHAMMAD-638573336274598086_15.npy
✅ Converted: NORAEI SHAHINDOKHT-638839486874254842_15.npy
✅ Converted: BOLANDHEMAT ALI.MOHAMAD-638573387891883483_4.npy
✅ Converted: 67_29.npy
✅ Converted: HAMZEEI FARIDE-638573365169189689_26.npy
✅ Converted: SHAMSAEI MEHDI-638839513134388151_20.npy
✅ Converted: PARSEH DAVOD-638839478192330005_36.npy
✅ Converted: 13_40.npy
✅ Converted: POURMOMEN MARZIYEH-638573305944410911_35.npy
✅ Converted: ASGARIYAN MOHAMADHOSSEIN-638573324701702776_19.npy
✅ Converted: NIKFARMAN NARGES-638573270048500219_34.npy
✅ Converted: HAJIVAND GHOLAM ALI-638573381195936947_4.npy
✅ Converted: mahmodi mohamad reza-638573320540559512_11.npy
✅ Converted: NAJAFI-638573264420442728_12.npy
✅ Converted: TABARSAEI MOHAMMAD-638573338734722393_17.npy
✅ Converted: TAJIK MOJ

# Splitting

In [14]:
# Updated CSV paths
train_csv = "/kaggle/input/hemorica-splits/patient_wise_train_split.csv"
val_csv = "/kaggle/input/hemorica-splits/patient_wise_val_split.csv"
test_csv = "/kaggle/input/hemorica-splits/test_HemoricaV1_seed51.csv"

# Base directories (update these as needed)
image_dir = "/kaggle/working/Working/images"
label_dir = "/kaggle/working/Working/labels"

# Destination directories
train_image_dir = "/kaggle/working/dataset/images/train"
train_label_dir = "/kaggle/working/dataset/labels/train"
val_image_dir   = "/kaggle/working/dataset/images/val"
val_label_dir   = "/kaggle/working/dataset/labels/val"
test_image_dir  = "/kaggle/working/dataset/images/test"
test_label_dir  = "/kaggle/working/dataset/labels/test"

# Make destination folders
for path in [train_image_dir, train_label_dir, val_image_dir, val_label_dir, test_image_dir, test_label_dir]:
    os.makedirs(path, exist_ok=True)

# Function to move files based on dataframe and fold type
def move_files(df, fold_type='train'):
    for _, row in df.iterrows():
        patient_name = row['patient_name']
        slice_number = row['slice_number']
        image_filename = f"{patient_name}_{slice_number}.png"
        label_filename = f"{patient_name}_{slice_number}.txt"

        source_image = os.path.join(image_dir, image_filename)
        source_label = os.path.join(label_dir, label_filename)

        if not os.path.exists(source_image) or not os.path.exists(source_label):
            print(f"⚠️ Skipping {image_filename} - missing file.")
            continue

        if fold_type == 'train':
            dest_img, dest_lbl = train_image_dir, train_label_dir
        elif fold_type == 'val':
            dest_img, dest_lbl = val_image_dir, val_label_dir
        else:
            dest_img, dest_lbl = test_image_dir, test_label_dir

        shutil.copy2(source_image, os.path.join(dest_img, image_filename))
        shutil.copy2(source_label, os.path.join(dest_lbl, label_filename))
        print(f"✅ Copied {image_filename} and label to {fold_type} set.")

# Read and move files for each split
move_files(pd.read_csv(train_csv), fold_type='train')
move_files(pd.read_csv(val_csv), fold_type='val')
move_files(pd.read_csv(test_csv), fold_type='test')

print("\n✅ Dataset split into train/val/test sets successfully.")

✅ Copied 67_0.png and label to train set.
✅ Copied 67_1.png and label to train set.
✅ Copied 67_2.png and label to train set.
✅ Copied 67_3.png and label to train set.
✅ Copied 67_4.png and label to train set.
✅ Copied 67_5.png and label to train set.
✅ Copied 67_6.png and label to train set.
✅ Copied 67_7.png and label to train set.
✅ Copied 67_8.png and label to train set.
✅ Copied 67_9.png and label to train set.
✅ Copied 67_10.png and label to train set.
✅ Copied 67_11.png and label to train set.
✅ Copied 67_12.png and label to train set.
✅ Copied 67_13.png and label to train set.
✅ Copied 67_14.png and label to train set.
✅ Copied 67_15.png and label to train set.
✅ Copied 67_16.png and label to train set.
✅ Copied 67_17.png and label to train set.
✅ Copied 67_18.png and label to train set.
✅ Copied 67_19.png and label to train set.
✅ Copied 67_20.png and label to train set.
✅ Copied 67_21.png and label to train set.
✅ Copied 67_22.png and label to train set.
✅ Copied 67_23.png an

In [15]:
!zip -r /kaggle/working/Rasoul-MulticlassAnnotations.zip /kaggle/working/dataset/labels

  adding: kaggle/working/dataset/labels/ (stored 0%)
  adding: kaggle/working/dataset/labels/train/ (stored 0%)
  adding: kaggle/working/dataset/labels/train/SHAHMIRZAI PARVIN-638839450246490138_19.txt (deflated 27%)
  adding: kaggle/working/dataset/labels/train/LASHGARI AZAM-638839461907107237_21.txt (deflated 27%)
  adding: kaggle/working/dataset/labels/train/SHARIATI MASOOD-638573299427214142_28.txt (stored 0%)
  adding: kaggle/working/dataset/labels/train/57_18.txt (stored 0%)
  adding: kaggle/working/dataset/labels/train/MADANI GHOLAMREZA-638573381463136795_9.txt (deflated 27%)
  adding: kaggle/working/dataset/labels/train/21_13.txt (stored 0%)
  adding: kaggle/working/dataset/labels/train/BABAYEI NAZANINZAHRA-638839501066398891_23.txt (stored 0%)
  adding: kaggle/working/dataset/labels/train/NAGHI LOO MAJID-638573343277154088_19.txt (stored 0%)
  adding: kaggle/working/dataset/labels/train/55_10.txt (stored 0%)
  adding: kaggle/working/dataset/labels/train/26_9.txt (stored 0%)
  

In [16]:
!zip -r /kaggle/working/Rasoul-MulticlassDataset.zip /kaggle/working/dataset

  adding: kaggle/working/dataset/ (stored 0%)
  adding: kaggle/working/dataset/images/ (stored 0%)
  adding: kaggle/working/dataset/images/train/ (stored 0%)
  adding: kaggle/working/dataset/images/train/AFIFIMOGHADAM MOHAMAD HOSEIN-638573323614152553_28.png (deflated 13%)
  adding: kaggle/working/dataset/images/train/55_25.png (deflated 1%)
  adding: kaggle/working/dataset/images/train/DEHGHAN MOHAMAD GHORBAN-638573318279361312_0.png (deflated 2%)
  adding: kaggle/working/dataset/images/train/KHODAREZAYEI ALI-638573394962999855_1.png (deflated 1%)
  adding: kaggle/working/dataset/images/train/KOKABI SAEED-638839515739345864_30.png (deflated 14%)
  adding: kaggle/working/dataset/images/train/MOAZEZI FASTEMEH-638573379246315706_25.png (deflated 2%)
  adding: kaggle/working/dataset/images/train/AKBARI ALIREZA-638573386428065874_18.png (deflated 5%)
  adding: kaggle/working/dataset/images/train/MMIRZAI MAHDI-638573281732043269_17.png (deflated 1%)
  adding: kaggle/working/dataset/images/t

In [17]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

def visualize_yolo_obb(image_path, label_path):
    """
    Visualizes YOLOv8 OBB annotations on a given image.
    
    Args:
        image_path (str): Path to the image.
        label_path (str): Path to the corresponding YOLO OBB label file.
    """
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Failed to load image: {image_path}")
        return
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]

    # Read label file
    if not os.path.exists(label_path):
        print(f"❌ Label file not found: {label_path}")
        return

    with open(label_path, 'r') as f:
        lines = f.readlines()

    # Draw OBBs
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 9:
            continue
        class_id = int(parts[0])
        coords = np.array([float(p) for p in parts[1:]], dtype=np.float32).reshape(-1, 2)
        coords[:, 0] *= width
        coords[:, 1] *= height
        coords = coords.astype(np.int32)

        cv2.polylines(image, [coords], isClosed=True, color=(255, 0, 0), thickness=2)
        cx = int(coords[:, 0].mean())
        cy = int(coords[:, 1].mean())
        cv2.putText(image, str(class_id), (cx, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    # Show image
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title("YOLOv8 OBB Annotations")
    plt.show()

In [18]:
destination_dir = "/kaggle/working/dataset"

def count_files_in_directory(directory):
    total_files = 0
    # Iterate through each item in the directory
    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)
        # If the item is a directory, recursively count files in it
        if os.path.isdir(item_path):
            num_files_in_subfolder = count_files_in_directory(item_path)
            print(f"Folder '{item}': {num_files_in_subfolder} files")
            total_files += num_files_in_subfolder
        # If the item is a file, increment the file count
        elif os.path.isfile(item_path):
            total_files += 1
    return total_files

# Check if the destination directory exists
if os.path.exists(destination_dir):
    total_files = count_files_in_directory(destination_dir)
    print(f"Total files in '{destination_dir}': {total_files}")
else:
    print(f"The directory '{destination_dir}' does not exist.")

Folder 'train': 7648 files
Folder 'val': 1990 files
Folder 'test': 2429 files
Folder 'images': 12067 files
Folder 'train': 7648 files
Folder 'val': 1990 files
Folder 'test': 2429 files
Folder 'labels': 12067 files
Total files in '/kaggle/working/dataset': 24134


In [19]:
visualize_yolo_obb(
    image_path="/kaggle/working/dataset/images/test/NOROOZI MOHAMAD MAHDI-638573317026490922_15.png",
    label_path="/kaggle/working/dataset/labels/test/NOROOZI MOHAMAD MAHDI-638573317026490922_15.txt"
)

❌ Failed to load image: /kaggle/working/dataset/images/test/NOROOZI MOHAMAD MAHDI-638573317026490922_15.png


[ WARN:0@741.643] global loadsave.cpp:268 findDecoder imread_('/kaggle/working/dataset/images/test/NOROOZI MOHAMAD MAHDI-638573317026490922_15.png'): can't open/read file: check file path/integrity
